# **Vision-Based Sense and Avoid**

| Role          | Name / Details                                                  |
| ------------- | --------------------------------------------------------------- |
| Author        | Max Busato                                                      |
| Sponsor       | Lockheed Martin Australia – Advanced Systems & Technology Group |
| Supervisor    | Dr Feras Dayoub                                                 |
| Co-Supervisor | Ms Kendall Jenner                                               |


## **Monocular Airborne Object Tracking for Safe Drone Flight**

This notebook provides an end-to-end demonstration of the final trained model for this project, covering dataset download, model weight loading, and result production on the custom version of the Airborne Object Tracking (AOT) dataset. The AOT dataset is a real-world collection of 5.9M+ images across diverse flight conditions, where airborne objects can appear as just a few pixels per frame, making this one of the most challenging small-object detection problems in computer vision, and one of the most critical for future autonomous flight systems.

*For full project setup, dataset details, and source scripts see the [Project README](../README.md).*

In [ ]:
# Import Required Libraries.
import os
import sys

# Set Working Directory to Project Root.
os.chdir("..") 

# Add Project Root to Path.
sys.path.insert(0, os.getcwd())


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets** - Airborne objects occupying only a few pixels per frame.
- **Multiple Targets** - Scenes containing more than one airborne object simultaneously.
- **Poor Visibility** - Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Import Data Downloader Libraries.
from src.data.download_data import main as downloadData

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"

# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)

In [1]:
# ------------------------------------------------------------------------------
#      FIND THE LOWEST FTPS THRESHOLD THAT STILL GIVES ZERO FALSE TRACKS
# ------------------------------------------------------------------------------
# For every model under outputs/results/core_improvement, this locates the
# latest run, reads that run's per-track FTPS breakdown (*_ftps_tracks.csv,
# produced by compute_ftps() in calculate_airborne_metrics.py), and reports
# the worst (highest) unmatched_ratio across all eligible tracks.
#
# That value IS the threshold boundary: any FTPS threshold set ABOVE it will
# report 0 false tracks; any threshold AT or BELOW it will start catching
# that track. This is purely a metrics-side query -- no inference is re-run,
# since everything needed already exists in each run's saved CSV output.

# Import Required Libraries.
import os
import re
import sys
import random

import pandas as pd
from glob import glob
from logging_module import LogLevel, log

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))
 
# Confirm Project Root.
log(LogLevel.INFO, "core_improvement:imports", "Project Root Set: %s", PROJECT_ROOT)

# Root Directory Containing Every Model's Results.
RESULTS_ROOT = "outputs/results/core_improvement"

# Reported Thresholds, for Comparison Against the Computed Floor.
REPORTED_THRESHOLDS = [0.50, 0.80, 0.95]


def find_latest_run_dir(model_dir: str) -> str | None:
    """
    Finds the Latest 'runN' Folder Inside a Model's Results Directory.

    Matches the outputs/results/core_improvement/<model>/run4/... convention.
    Picks the run with the highest numeric suffix. Falls back to the most
    recently modified folder if no run folders match the expected pattern.
    """
    run_dirs = glob(os.path.join(model_dir, "run*"))
    run_dirs = [d for d in run_dirs if os.path.isdir(d)]

    if not run_dirs:
        return None

    def _run_number(path):
        match = re.search(r"run(\d+)$", os.path.basename(path))
        return int(match.group(1)) if match else -1

    numbered_runs = [d for d in run_dirs if _run_number(d) >= 0]
    if numbered_runs:
        return max(numbered_runs, key=_run_number)

    # Fallback: No Parsable "runN" Names Found, Use Most Recently Modified.
    return max(run_dirs, key=os.path.getmtime)


def find_ftps_tracks_csv(run_dir: str) -> str | None:
    """
    Recursively Searches a Run Directory for the Per-Track FTPS Breakdown CSV.

    The exact subfolder (e.g. summaries/) can vary by run, so this searches
    the whole run directory tree rather than assuming a fixed layout.
    """
    matches = glob(os.path.join(run_dir, "**", "*_ftps_tracks.csv"), recursive=True)
    if not matches:
        return None
    # If Multiple Match (Different Score Thresholds Evaluated), Use the Most Recent.
    return max(matches, key=os.path.getmtime)


# Discover Every Model Folder Under the Results Root.
if not os.path.isdir(RESULTS_ROOT):
    print(f"Results root not found: {RESULTS_ROOT}")
    model_dirs = []
else:
    model_dirs = sorted(
        d for d in glob(os.path.join(RESULTS_ROOT, "*"))
        if os.path.isdir(d)
    )

# Collect One Row of Findings per Model.
rows = []

for model_dir in model_dirs:
    model_name = os.path.basename(model_dir)

    latest_run = find_latest_run_dir(model_dir)
    if latest_run is None:
        rows.append({
            "Model": model_name, "Latest Run": "N/A",
            "Lowest Safe Threshold": "N/A",
            "Note": "No run folders found",
        })
        continue

    ftps_csv = find_ftps_tracks_csv(latest_run)
    if ftps_csv is None:
        rows.append({
            "Model": model_name, "Latest Run": os.path.basename(latest_run),
            "Lowest Safe Threshold": "N/A",
            "Note": "No FTPS metrics found -- run calculate_airborne_metrics.py for this run first",
        })
        continue

    df = pd.read_csv(ftps_csv)

    # No Eligible Tracks at All (e.g. Model Produced Nothing Meeting min_track_frames).
    if df.empty or "unmatched_ratio" not in df.columns:
        rows.append({
            "Model": model_name, "Latest Run": os.path.basename(latest_run),
            "Lowest Safe Threshold": "N/A",
            "Note": "No eligible tracks in this run",
        })
        continue

    # The Worst (Highest) Unmatched Ratio IS the Threshold Boundary.
    # Any threshold strictly ABOVE this value guarantees 0 false tracks.
    worst_ratio = df["unmatched_ratio"].max()

    # Flag Whether This Model Already Has False Tracks at the Reported Thresholds.
    already_flagged_at = [t for t in REPORTED_THRESHOLDS if worst_ratio >= t]
    if already_flagged_at:
        note = f"Already >= 0 false tracks at reported threshold(s): {already_flagged_at}"
    else:
        note = "Clean at all reported thresholds (50/80/95) -- floor shown is the true boundary"

    rows.append({
        "Model": model_name,
        "Latest Run": os.path.basename(latest_run),
        "Lowest Safe Threshold": f"{worst_ratio:.4f}",
        "Note": note,
    })

# Display Results as a Table.
df_summary = pd.DataFrame(rows)
display(df_summary)
# ------------------------------------------------------------------------------

<INFO ><core_improvement:imports> Project Root Set: /home/maxbusato/Projects/Vision-Based-Sense-and-Avoid


,Model,Latest Run,Lowest Safe Threshold,Note
0,efficientvitb2,run3,1.0000,Already >= 0 false tracks at reported threshol...
1,efficientvitm2,run3,0.0286,Clean at all reported thresholds (50/80/95) --...
2,hrnetw32,run13,0.0021,Clean at all reported thresholds (50/80/95) --...
3,hrnetw32_track_to_detect,run2,1.0000,Already >= 0 false tracks at reported threshol...
4,mobilenetv4,run3,0.0312,Clean at all reported thresholds (50/80/95) --...
5,resnet34,run3,1.0000,Already >= 0 false tracks at reported threshol...
6,winning_solution,run1,N/A,No FTPS metrics found -- run calculate_airborn...
7,winning_solution_star,run3,0.0030,Clean at all reported thresholds (50/80/95) --...



---

<br/>
